
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
# Init model
model = init_chat_model("groq:qwen/qwen3.6-27b")

In [3]:
from pydantic import BaseModel,Field

# Init structured output for movie

# Tên Structure (Tên Class) 
# Khai báo các thông tin trả về như thuộc tính của Structure 
# Thông tin: tên thông tin, kiểu dữ liệu, mô tả (để LLM hiểu là phải trả về thông tin có đặc điểm như nào)

class Movie(BaseModel): 
    title:str = Field(description="Tiêu đề/tên của bộ phim") 
    year:int = Field(description="Năm bộ phim được xuất bản")
    director:str = Field(description="Đạo diễn của bộ phim")
    rating:float = Field(description="Điểm đánh giá của bộ phim") 

In [31]:
# Result

In [5]:
# Compare LLM with & without structured output
# Hãy cung cấp cho tôi thông tin về bộ phim Obsession
model.invoke("Hãy cung cấp cho tôi thông tin về bộ phim Inception")

AIMessage(content='\n<think>\nWe need to provide information about the movie "Inception". The user asked in Vietnamese: "Hãy cung cấp cho tôi thông tin về bộ phim Inception". So we should respond in Vietnamese.\n\nWe need to give accurate information about Inception: director, release year, cast, plot summary, awards, etc. But we should keep it concise and in Vietnamese.\n\nLet me recall: Inception is a 2010 science fiction action film written and directed by Christopher Nolan. It stars Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page, Tom Hardy, Ken Watanabe, Cillian Murphy, Marion Cotillard, Michael Caine. The plot involves dream sharing technology and a team that performs inception (planting an idea) in someone\'s subconscious. It won four Academy Awards, including Best Cinematography, Best Sound Mixing, Best Sound Editing, Best Visual Effects. It was nominated for Best Picture, Best Original Screenplay, Best Art Direction, Best Original Score.\n\nWe should present this in Vietn

In [ ]:
model_with_structured_output = model.with_structured_output(Movie) 
model_with_structured_output.invoke("Hãy cung cấp cho tôi thông tin về bộ phim Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [34]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Identify User Request**: The user wants details about the movie "Inception".\n2.  **Identify Available Tool**: The `Movie` tool can provide details about a movie. It requires `title`, `year`, `director`, and `rating`.\n3.  **Gather Information for Tool**:\n   - Title: "Inception"\n   - Year: 2010\n   - Director: Christopher Nolan\n   - Rating: 8.8 (widely known IMDb rating, but I should verify or use a standard accepted rating. I\'ll use 8.8 or 8.8/10)\n4.  **Call Tool**: `Movie(title="Inception", year=2010, director="Christopher Nolan", rating=8.8)`\n5.  **Process Output**: The tool will return the movie details. I will then present them to the user.\n   - Wait, the tool definition says it\'s a function to *provide* a movie with details, but it takes all parameters as input. This is a bit unusual for a typical API (usually you query by title and get back details), but I must follow the sche

### Nested Structure

In [7]:
from pydantic import BaseModel, Field

# Init Actor (name, role)
class Actor(BaseModel): 
    name: str 
    role: str 

# Init MovieDetails (title, year, cast, genres, budget)
class MovieDetails(BaseModel): 
    title: str 
    year: int 
    cast: list[Actor] 
    genres: list[str] 
    budget: float | None 

In [8]:
movie_details_model = model.with_structured_output(MovieDetails) 
response = movie_details_model.invoke("Hãy cung cấp cho tôi thông tin về bộ phim Inception")

In [9]:
cast_lst = [actor.name for actor in response.cast]
cast_lst 

['Leonardo DiCaprio',
 'Joseph Gordon-Levitt',
 'Elliot Page',
 'Tom Hardy',
 'Ken Watanabe',
 'Marion Cotillard',
 'Cillian Murphy']

In [10]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [11]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

In [12]:
agent = create_agent(
    model="gpt-5-mini", 
    response_format=ContactInfo 
)

In [13]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Extract contact info from Tom Cruise"}
    ]
})

result

{'messages': [HumanMessage(content='Extract contact info from Tom Cruise', additional_kwargs={}, response_metadata={}, id='b3281911-795e-4958-82a0-0fc350235ada'),
  AIMessage(content='{"name":"Tom Cruise","email":"","phone":""}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 665, 'prompt_tokens': 190, 'total_tokens': 855, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E3HIbMMoZOFs9ISi5x5WPS5EU1N72', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f7984-f5db-71a3-8d6a-d6f2e12c3f16-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 190, 'output_tokens': 665, 'total_tokens